# Bell State Preparation experiment: Compute $\langle \bar{\Phi}^+ | \bar{Z}\bar{Z} | \bar{\Phi}^+\rangle$ with the seven-qubit Steane code

In [1]:
from typing import List, Dict, Sequence
import itertools
import functools
import numpy as np
from tqdm import tqdm
import cirq
import qiskit
from qiskit.circuit.library import Barrier
import qiskit_ibm_runtime
from qiskit_ibm_runtime import SamplerV2 as Sampler

from encoded.diagonalize import get_stabilizer_matrix_from_paulis, get_measurement_circuit, get_paulis_from_stabilizer_matrix, string_to_cirq_paulistring
from encoded.tcc import tcc_encoding

/Users/ryan/prof/work/encoded/envencoded/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
import datetime


time_key = datetime.datetime.now().strftime("%m_%d_%Y_%H:%M:%S")  # For saving results.

## Set parameters

In [3]:
d = 3                                   # Code distance.
nshots = 10_000                         # Number of samples/shots
depth = 0                               # Number of folded Bell state preparation circuits for added noise
k = 2                                   # Number of logical qubits.

In [4]:
distance_to_nqubits = lambda d: (3 * d ** 2 + 1) // 4

n = distance_to_nqubits(d)

In [5]:
# Computer and qubits to use.
service = qiskit_ibm_runtime.QiskitRuntimeService(
    channel="ibm_cloud",
    token="Vwp-noJ99qjdtslYdRY6aEClfwr6PM63bP1Az7fE3015",
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/8adb079e32e34c679acff94b68640033:f54da97d-7fe5-4108-9409-98a68f6ca53f::",
)
computer = service.backend("ibm_boston")
sampler = Sampler(computer)

# See calibration data at https://quantum.ibm.com/services/resources to select good qubits.
layout = {
    1: [2, 3],
    3: [2, 3, 4, 5, 6, 7, 16, 17, 22, 23, 24, 25, 26, 27],
    5: list(range(38)),
}


# The best qubits ever on Fez Feb 6-8.
# layout = {
#     1 : [123, 124],
#     7 : [123, 124, 125, 126, 127, 128, 136, 137, 142, 143, 144, 145, 146, 147],
# }


# Good on Kyiv 2/6
# layout = {
#     1 : [20, 21],
#     7 : [0, 1, 2, 3, 4, 5, 14, 15, 18, 19, 20, 21, 22, 23],
# }

# Good on Sherbrooke 2/5.
# layout = {
#     1 : [103, 104],
#     7 : [103, 104, 105, 106, 107, 108, 111, 112, 121, 122, 123, 124, 125, 126],
# }

# Good on Kyiv 2/5.
# layout = {
#     1 : [95, 96],
#     7 : [95, 96, 97, 98, 99, 100, 101, 113, 114, 115, 116, 117, 118, 119],
# }

qiskit_runtime_service._discover_account:WARNING:2026-04-12 13:05:00,171: Loading account with the given token. A saved account will not be used.


## Helper functions

In [6]:
# Expectation of pauli on bitstring measured in diagonal basis.
def compute_expectation(
    pauli: cirq.PauliString,
    counts: Dict[str, int],
) -> float:
    if pauli is cirq.PauliString():
        return 1.0

    expectation = 0.0

    indices = [q.x for q in pauli.qubits]
    for key, value in counts.items():
        key = list(map(int, list(key[::-1])))
        expectation += (-1) ** sum([key[i] for i in indices]) * value

    return expectation / sum(counts.values())

# Prepares logical |0> state on Steane Code
def encode_steane(qreg: Sequence[cirq.Qid]) -> cirq.Circuit:
    circuit = cirq.Circuit()

    circuit.append(cirq.H.on(qreg[0]))
    circuit.append(cirq.H.on(qreg[4]))
    circuit.append(cirq.H.on(qreg[6]))

    circuit.append(cirq.CNOT.on(qreg[0], qreg[1]))
    circuit.append(cirq.CNOT.on(qreg[4], qreg[5]))

    circuit.append(cirq.CNOT.on(qreg[6], qreg[3]))
    circuit.append(cirq.CNOT.on(qreg[6], qreg[5]))
    circuit.append(cirq.CNOT.on(qreg[4], qreg[2]))
    
    circuit.append(cirq.CNOT.on(qreg[0], qreg[3]))
    circuit.append(cirq.CNOT.on(qreg[4], qreg[1]))
    circuit.append(cirq.CNOT.on(qreg[3], qreg[2]))

    return circuit

def strs_to_paulis(pauli_strs : List[str]) -> List[cirq.PauliString]:
    stab_list = []
    for stab_str in pauli_strs:
        stab_list.append(string_to_cirq_paulistring(stab_str))
    return stab_list

def generate_stabilizer_elements(generators: List[cirq.PauliString]) -> List[cirq.PauliString]:
    elements = []
    for string in itertools.chain.from_iterable(itertools.combinations(generators, r) for r in range(len(generators) + 1)):
        elements.append(
            functools.reduce(lambda a, b: a * b, string, cirq.PauliString())
        )
    return elements

# For qiskit circuits
def get_active_qubits(circ):
    dag = qiskit.converters.circuit_to_dag(circ)
    active_qubits = [qubit for qubit in circ.qubits if qubit not in dag.idle_wires()]
    return active_qubits

def get_lst_ev(counts, observables, stabilizers):
    numerator = 0
    for obs in observables:
        numerator += compute_expectation(obs, counts) / len(observables)
    denominator = 0
    for stab in stabilizers:
        denominator += compute_expectation(stab, counts) / len(stabilizers)
    return float(np.real_if_close(numerator / denominator))


### Run unmitigated experiment

In [7]:
qreg = cirq.LineQubit.range(k)

circuit = cirq.Circuit()
circuit.append(cirq.H.on(qreg[0]))
for i in range(len(qreg)-1):
    circuit.append(cirq.CNOT.on(qreg[i], qreg[i+1]))

circuit = qiskit.QuantumCircuit.from_qasm_str(circuit.to_qasm())
circuit.measure_active()
# Compile to device.
compiled_physical = qiskit.transpile(
    circuit, 
    backend=computer,
    initial_layout=layout[1],  # Hardcode n = 1 (i.e., no encoding) to get layout.
    routing_method="sabre",
    # scheduling_method="asap",
    optimization_level=0,
)
print(compiled_physical.draw(fold=-1, idle_wires=False))

global phase: 3π/4
         ┌─────────┐┌────┐┌─────────┐                                ░ ┌─┐   
q_0 -> 2 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─■──────────────────────────────░─┤M├───
         ├─────────┤├────┤├─────────┤ │ ┌─────────┐┌────┐┌─────────┐ ░ └╥┘┌─┐
q_1 -> 3 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─░──╫─┤M├
         └─────────┘└────┘└─────────┘   └─────────┘└────┘└─────────┘ ░  ║ └╥┘
 meas: 2/═══════════════════════════════════════════════════════════════╩══╩═
                                                                        0  1 


In [ ]:
job_physical = sampler.run(
    [compiled_physical],
    shots=nshots
)

In [ ]:
all_counts_raw = [result.data.meas.get_counts() for result in job_physical.result()]
all_counts_raw

In [ ]:
ev_physical = compute_expectation(string_to_cirq_paulistring("ZZ"), all_counts_raw[0])
print("Physical result:", ev_physical)

## Run encoded experiment

In [9]:
correct_bitstrings = ["0000000", "1010101", "0110011", "1100110", "0001111", "1011010", "0111100", "1101001"]

In [20]:
# Current solution.
ec = encode_steane(cirq.LineQubit.range(7))
dirac = cirq.dirac_notation(ec.final_state_vector(initial_state=0))
for correct_bitstring in correct_bitstrings:
    if correct_bitstring in dirac:
        print(correct_bitstring)
    else:
        print("Missing", correct_bitstring)

0000000
1010101
Missing 0110011
Missing 1100110
Missing 0001111
Missing 1011010
Missing 0111100
Missing 1101001


In [28]:
import stim
import stimcirq

In [82]:
stim.Tableau.from_state_vector(state_vector=np.ones(2 ** 23) / np.sqrt(2 ** 23), endian="big").to_circuit()

stim.Circuit('''
    H 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22
''')

In [162]:
code_generators = [stim.PauliString("XX_"), stim.PauliString("_XX"), stim.PauliString("ZZZ")]
stimcirq.stim_circuit_to_cirq_circuit(stim.Tableau.from_stabilizers(code_generators).to_circuit()).final_state_vector()

array([0.5+0.j, 0. +0.j, 0. +0.j, 0.5+0.j, 0. +0.j, 0.5+0.j, 0.5+0.j,
       0. +0.j])

In [110]:
import itertools
import functools

In [126]:
def compute_stabilizer_elements_from_stabilizer_generators(
        code_generators: List[stim.PauliString]
) -> List[stim.PauliString]:
    num_generators = len(code_generators)
    code_elements = []
    for t in range(2 ** num_generators):
        exponents = [int(b) for b in np.binary_repr(t, width=len(code_generators))]
        to_multiply = [generator if b == 1 else stim.PauliString() for generator, b in zip(code_generators, exponents)]
        element = functools.reduce(lambda a, b: a * b, to_multiply)
        code_elements.append(element)
    return code_elements

In [135]:
stabilizers = compute_stabilizer_elements_from_stabilizer_generators(code_generators)

tab = stim.Tableau.from_stabilizers(stabilizers, allow_underconstrained=True, allow_redundant=True)

ec = stimcirq.stim_circuit_to_cirq_circuit(tab.to_circuit())
cirq.dirac_notation(ec.final_state_vector())

'0.5|000⟩ + 0.5|011⟩ + 0.5|101⟩ + 0.5|110⟩'

In [88]:
stim.Tableau?

Docstring:     
A stabilizer tableau.

Represents a Clifford operation by explicitly storing how that operation
conjugates a list of Pauli group generators into composite Pauli products.

Examples:
    >>> import stim
    >>> stim.Tableau.from_named_gate("H")
    stim.Tableau.from_conjugated_generators(
        xs=[
            stim.PauliString("+Z"),
        ],
        zs=[
            stim.PauliString("+X"),
        ],
    )

    >>> t = stim.Tableau.random(5)
    >>> t_inv = t**-1
    >>> print(t * t_inv)
    +-xz-xz-xz-xz-xz-
    | ++ ++ ++ ++ ++
    | XZ __ __ __ __
    | __ XZ __ __ __
    | __ __ XZ __ __
    | __ __ __ XZ __
    | __ __ __ __ XZ

    >>> x2z3 = t.x_output(2) * t.z_output(3)
    >>> t_inv(x2z3)
    stim.PauliString("+__XZ_")
Init docstring:
__init__(self: stim._stim_polyfill.Tableau, num_qubits: int) -> None

Creates an identity tableau over the given number of qubits.

Examples:
    >>> import stim
    >>> t = stim.Tableau(3)
    >>> print(t)
    +-xz-xz-xz-
  

In [153]:
stim.Tableau.from_stabilizers?

Docstring:
from_stabilizers(stabilizers: object, *, allow_redundant: bool = False, allow_underconstrained: bool = False) -> stim._stim_polyfill.Tableau

@signature def from_stabilizers(stabilizers: Iterable[stim.PauliString], *, allow_redundant: bool = False, allow_underconstrained: bool = False) -> stim.Tableau:
Creates a tableau representing a state with the given stabilizers.

Args:
    stabilizers: A list of `stim.PauliString`s specifying the stabilizers that
        the state must have. It is permitted for stabilizers to have different
        lengths. All stabilizers are padded up to the length of the longest
        stabilizer by appending identity terms.
    allow_redundant: Defaults to False. If set to False, then the given
        stabilizers must all be independent. If any one of them is a product of
        the others (including the empty product), an exception will be raised.
        If set to True, then redundant stabilizers are simply ignored.
    allow_underconstrained:

In [151]:
stim.Tableau.from_named_gate("CZ")

stim.Tableau.from_conjugated_generators(
    xs=[
        stim.PauliString("+XZ"),
        stim.PauliString("+ZX"),
    ],
    zs=[
        stim.PauliString("+Z_"),
        stim.PauliString("+_Z"),
    ],
)

In [145]:
stim.Tableau.to_state_vector?

Call signature:  stim.Tableau.to_state_vector(*args, **kwargs)
Type:            instancemethod
String form:     <instancemethod to_state_vector at 0x11b5f5c30>
Docstring:      
to_state_vector(self: stim._stim_polyfill.Tableau, *, endian: str = 'little') -> numpy.ndarray[numpy.complex64]

@signature def to_state_vector(self, *, endian: str = 'little') -> np.ndarray[np.complex64]:
Returns the state vector produced by applying the tableau to the |0..0> state.

This function takes O(n * 2**n) time and O(2**n) space, where n is the number of
qubits. The computation is done by initialization a random state vector and
iteratively projecting it into the +1 eigenspace of each stabilizer of the
state. The state is then canonicalized so that zero values are actually exactly
0, and so that the first non-zero entry is positive.

Args:
    endian:
        "little" (default): state vector is in little endian order, where higher
            index qubits correspond to larger changes in the state index

In [92]:
tab.to_stabilizers?

Docstring:
to_stabilizers(self: stim._stim_polyfill.Tableau, *, canonicalize: bool = False) -> List[stim._stim_polyfill.PauliString]

Returns the stabilizer generators of the tableau, optionally canonicalized.

The stabilizer generators of the tableau are its Z outputs. Canonicalizing
standardizes the generators, so that states that are equal will produce the
same generators. For example, [ZI, IZ], [ZI, ZZ], amd [ZZ, ZI] describe equal
states and all canonicalize to [ZI, IZ].

The canonical form is computed as follows:

    1. Get a list of stabilizers using `tableau.z_output(k)` for each k.
    2. Perform Gaussian elimination. pivoting on standard generators.
        2a) Pivot on g=X0 first, then Z0, X1, Z1, X2, Z2, etc.
        2b) Find a stabilizer that uses the generator g. If there are none,
            go to the next g.
        2c) Multiply that stabilizer into all other stabilizers that use the
            generator g.
        2d) Swap that stabilizer with the stabilizer at posi

In [95]:
tab.from_stabilizers(code_generators, allow_underconstrained=True).to_stabilizers()

[stim.PauliString("+XX_"), stim.PauliString("+_XX"), stim.PauliString("+ZZZ")]

In [54]:
tableau = stim.Tableau.from_stabilizers(
    [stim.PauliString("XX_"), stim.PauliString("_XX")], 
    allow_underconstrained=True,
)
tableau

stim.Tableau.from_conjugated_generators(
    xs=[
        stim.PauliString("+Z__"),
        stim.PauliString("+ZZ_"),
        stim.PauliString("+__X"),
    ],
    zs=[
        stim.PauliString("+XX_"),
        stim.PauliString("+_XX"),
        stim.PauliString("+ZZZ"),
    ],
)

In [83]:
encoding = stimcirq.stim_circuit_to_cirq_circuit(
    tableau.to_circuit()
)
encoding

0: ───H───@───@───────
          │   │
1: ───H───X───┼───@───
              │   │
2: ───────────X───X───

In [84]:
cirq.dirac_notation(encoding.final_state_vector())

'0.5|000⟩ + 0.5|011⟩ + 0.5|101⟩ + 0.5|110⟩'

In [164]:
encode = tcc_encoding(3)
encode

encode = cirq.H.on_each(encode.all_qubits()) + encode[1:]
state = encode.final_state_vector(initial_state=0)
dirac = cirq.dirac_notation(state)
for correct_bitstring in correct_bitstrings:
    if correct_bitstring in dirac:
        print(correct_bitstring)
    else:
        print("Missing", correct_bitstring)

0000000
1010101
Missing 0110011
Missing 1100110
Missing 0001111
Missing 1011010
Missing 0111100
Missing 1101001


In [166]:
l0 = state.copy()

In [174]:
l1 = stim.PauliString("XXXXXXX").to_unitary_matrix(endian="big") @ l0

/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/3687524558.py:1: RuntimeWarning: divide by zero encountered in matmul
  l1 = stim.PauliString("XXXXXXX").to_unitary_matrix(endian="big") @ l0
/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/3687524558.py:1: RuntimeWarning: overflow encountered in matmul
  l1 = stim.PauliString("XXXXXXX").to_unitary_matrix(endian="big") @ l0
/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/3687524558.py:1: RuntimeWarning: invalid value encountered in matmul
  l1 = stim.PauliString("XXXXXXX").to_unitary_matrix(endian="big") @ l0


In [187]:
assert np.isclose(l0 @ l1.conj().T, 0)

In [ ]:
assert np.allclose(stim.PauliString("ZZZZZZZ").to_unitary_matrix(endian="big") @ l0, l0)
assert np.allclose(stim.PauliString("ZZZZZZZ").to_unitary_matrix(endian="big") @ l1, -l1)
assert np.allclose(stim.PauliString("ZZZZZZZ").to_unitary_matrix(endian="big") @ l1, -l1)

/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/739731268.py:1: RuntimeWarning: divide by zero encountered in matmul
  assert np.allclose(stim.PauliString("ZZZZZZZ").to_unitary_matrix(endian="big") @ l0, l0)
/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/739731268.py:1: RuntimeWarning: overflow encountered in matmul
  assert np.allclose(stim.PauliString("ZZZZZZZ").to_unitary_matrix(endian="big") @ l0, l0)
/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/739731268.py:1: RuntimeWarning: invalid value encountered in matmul
  assert np.allclose(stim.PauliString("ZZZZZZZ").to_unitary_matrix(endian="big") @ l0, l0)
/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/739731268.py:2: RuntimeWarning: divide by zero encountered in matmul
  assert np.allclose(stim.PauliString("ZZZZZZZ").to_unitary_matrix(endian="big") @ l1, -l1)
/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/739731268.py:2: RuntimeWarning: overflow 

In [193]:
assert np.allclose(
    stim.Circuit("H 0 1 2 3 4 5 6").to_tableau().to_unitary_matrix(endian="big") @ l0, (l0 + l1) / np.sqrt(2)
)

/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/4165052256.py:2: RuntimeWarning: divide by zero encountered in matmul
  stim.Circuit("H 0 1 2 3 4 5 6").to_tableau().to_unitary_matrix(endian="big") @ l0, (l0 + l1) / np.sqrt(2)
/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/4165052256.py:2: RuntimeWarning: overflow encountered in matmul
  stim.Circuit("H 0 1 2 3 4 5 6").to_tableau().to_unitary_matrix(endian="big") @ l0, (l0 + l1) / np.sqrt(2)
/var/folders/z9/f3y13fq11gl5lnryg82b_ks40000gn/T/ipykernel_30033/4165052256.py:2: RuntimeWarning: invalid value encountered in matmul
  stim.Circuit("H 0 1 2 3 4 5 6").to_tableau().to_unitary_matrix(endian="big") @ l0, (l0 + l1) / np.sqrt(2)


In [136]:
import cirq

q = cirq.LineQubit.range(7)
c = cirq.Circuit(
    cirq.H(q[0]),
    cirq.H(q[1]),
    cirq.H(q[3]),
    cirq.CNOT(q[0], q[2]),
    cirq.CNOT(q[3], q[5]),
    cirq.CNOT(q[1], q[6]),
    cirq.CNOT(q[0], q[4]),
    cirq.CNOT(q[3], q[6]),
    cirq.CNOT(q[1], q[5]),
    cirq.CNOT(q[0], q[6]),
    cirq.CNOT(q[1], q[2]),
    cirq.CNOT(q[3], q[4]),
)
print(c)

state = c.final_state_vector(initial_state=0, qubit_order=q)
print(cirq.dirac_notation(state))

          ┌──┐   ┌───┐   ┌──┐
0: ───H────@──────@───────@─────
           │      │       │
1: ───H────┼@─────┼─@─────┼@────
           ││     │ │     ││
2: ────────X┼─────┼─┼─────┼X────
            │     │ │     │
3: ───H────@┼─────┼@┼─────┼@────
           ││     │││     ││
4: ────────┼┼─────X┼┼─────┼X────
           ││      ││     │
5: ────────X┼──────┼X─────┼─────
            │      │      │
6: ─────────X──────X──────X─────
          └──┘   └───┘   └──┘
0.35|0000000⟩ + 0.35|0001111⟩ + 0.35|0110011⟩ + 0.35|0111100⟩ + 0.35|1010101⟩ + 0.35|1011010⟩ + 0.35|1100110⟩ + 0.35|1101001⟩


In [139]:
et = stimcirq.stim_circuit_to_cirq_circuit(
    stim.Tableau.from_state_vector(state_vector=state, endian="big").to_circuit(method="elimination")
)
print(et)
np.allclose(et.final_state_vector(), state)

          ┌──┐           ┌──┐
0: ────────X────────────────────────────────────
           │
1: ────────┼─────X──────────────────────────────
           │     │
2: ────────┼─────┼───X────X─────────────────────
           │     │   │    │
3: ────────┼─────┼───┼────┼X────X───X───────────
           │     │   │    ││    │   │
4: ────────┼H────@───@────┼@────┼───┼───X───────
           │              │     │   │   │
5: ───H────@──────────────@─────@───┼───┼───X───
                                    │   │   │
6: ───────────────────────H─────────@───@───@───
          └──┘           └──┘


True

'0.35|0000000⟩ + 0.35|0011110⟩ + 0.35|0101101⟩ + 0.35|0110011⟩ + 0.35|1001011⟩ + 0.35|1010101⟩ + 0.35|1100110⟩ + 0.35|1111000⟩'

In [66]:
qubits = sorted(encode.all_qubits())
cirq.dirac_notation(encode.final_state_vector(initial_state=0, qubit_order=qubits))

'0.35|0000000⟩ + 0.35|0011011⟩ + 0.35|0101101⟩ + 0.35|0110110⟩ + 0.35|1001110⟩ + 0.35|1010101⟩ + 0.35|1100011⟩ + 0.35|1111000⟩'

In [ ]:
test = encode + cirq.measure(encode.all_qubits(), key="z")
test

In [ ]:
res = cirq.Simulator().run(test, repetitions=10)
res.measurements

In [ ]:
encode_circuit = tcc_encoding(d)
qubits = sorted(encode_circuit.all_qubits())

encode_circuit = cirq.H.on_each(qubits) + encode_circuit[1:]  # Remove RX gates to prepare the |+> state in favor of H.

qreg = cirq.LineQubit.range(n * k)
encoding = cirq.Circuit.concat_ragged(
    encode_circuit,
    encode_circuit.transform_qubits({a: b for a, b in zip(qreg[:n], qreg[n:])})
)

# Prepare Bell state
encoding.append(cirq.Moment(cirq.H.on_each(qreg[:n])))
encoding.append(cirq.CNOT.on_each([(qreg[i], qreg[i+n]) for i in range(n)]))
encoding

In [ ]:
result = cirq.Simulator().run(encoding + cirq.measure(encoding.all_qubits(), key="z"), repetitions=10)
result.data

In [ ]:
[int(np.binary_repr(int(l), n)[::-1], 2) for l in logical0]

In [ ]:
encoding = qiskit.QuantumCircuit.from_qasm_str(encoding.to_qasm())
encoding.draw(fold=-1)

In [ ]:
to_run = encoding.copy()
to_run.barrier()

# Idle time/X gates.
for _ in range(depth):
    to_run.x(to_run.qubits)
    to_run.barrier()
    to_run.x(to_run.qubits)
    to_run.barrier()

to_run.measure_all()

to_run.draw(fold=-1)

In [ ]:
to_run = qiskit.transpile(
    to_run, 
    backend=computer,
    initial_layout=layout[d],
    routing_method="sabre",
    # scheduling_method="asap",
    optimization_level=3,
)
to_run.draw(fold=-1, idle_wires=False)

In [ ]:
from qiskit_aer import AerSimulator

In [ ]:
job = AerSimulator().run(to_run, shots=nshots)

In [ ]:
res = job.result()
res.get_counts()

In [ ]:
logical0 = np.loadtxt("../../memory/hardware/logical0_d3.txt")
logical1 = np.loadtxt("../../memory/hardware/logical1_d3.txt")

In [ ]:
logical0s = [np.binary_repr(int(z), width=n)[::-1] for z in logical0]
logical1s = [np.binary_repr(int(z), width=n)[::-1] for z in logical1]

In [ ]:
from collections import Counter


logical_counts = Counter()
for ibm_counts in [res.get_counts()]:
    counts_qubit0 = Counter()
    counts_qubit1 = Counter()
    for bitstring, count in ibm_counts.items():
        qubit0 = bitstring[:n][::-1]
        qubit1 = bitstring[n:][::-1]

        if qubit0 in logical0s and qubit1 in logical0s:
            logical_counts["00"] += count
        elif qubit0 in logical0s and qubit1 in logical1s:
            logical_counts["01"] += count
        elif qubit0 in logical1s and qubit1 in logical0s:
            logical_counts["10"] += count
        elif qubit0 in logical1s and qubit1 in logical1s:
            logical_counts["11"] += count
        counts_qubit0[int(qubit0[::-1], 2)] += count
        counts_qubit1[int(qubit1[::-1], 2)] += count

In [ ]:
logical_counts

In [ ]:
all_counts_ints_qubit0

In [ ]:
all_counts_filtered = []
all_counts_filtered0 = []
all_counts_filtered1 = []

for counts_ints_qubit0, counts_ints_qubit1 in zip(all_counts_ints_qubit0, all_counts_ints_qubit1):
    filtered = Counter()
    filtered0 = Counter()
    filtered1 = Counter()

    for bitstring, count in counts_ints.items():
        if bitstring in logical0:
            filtered[bitstring] += count
            filtered0[bitstring] += count
        if bitstring in logical1:
            filtered[bitstring] += count
            filtered1[bitstring] += count
    all_counts_filtered.append(filtered)
    all_counts_filtered0.append(filtered0)
    all_counts_filtered1.append(filtered1)

In [ ]:
ncodewords = np.array([sum(filtered.values()) for filtered in all_counts_filtered])
ncodewords

In [ ]:
nzeroes = np.array([sum(filtered0.values()) for filtered0 in all_counts_filtered0])
nzeroes

In [ ]:
to_run.count_ops()

In [ ]:
job = sampler.run(
    [to_run],
    shots=nshots,
)

In [ ]:
all_counts = [result.data.measure.get_counts() for result in job.result()]

In [ ]:
ev = get_lst_ev(all_counts[0], tqdm(observable_elements), tqdm(stabilizer_elements))
print(ev)

## With DD

In [ ]:
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XpXm"

In [ ]:
job_dd = sampler.run(
    [compiled],
    shots=nshots,
)

In [ ]:
all_counts_dd = [result.data.measure.get_counts() for result in job_dd.result()]

In [ ]:
ev_dd = get_lst_ev(all_counts_dd[0], tqdm(observable_elements), tqdm(stabilizer_elements))
print(ev_dd)

## With REM + DD

In [ ]:
from qiskit_experiments.library import LocalReadoutError

In [ ]:
experiment = LocalReadoutError(layout[n])

In [ ]:
result = experiment.run(computer)

In [ ]:
mitigator = result.analysis_results("Local Readout Mitigator").value

In [ ]:
# for i, qubit in enumerate(mitigator.qubits):
#     print(f"Qubit: {qubit}")
#     print(mitigator.mitigation_matrix(qubits=i))

In [ ]:
counts = all_counts[0]
mitigated_quasi_probs = mitigator.quasi_probabilities(counts)
mitigated_stddev = mitigated_quasi_probs._stddev_upper_bound
mitigated_probs = (mitigated_quasi_probs.nearest_probability_distribution().binary_probabilities())

In [ ]:
mitigated_counts = {k: round(v * nshots) for k, v in mitigated_probs.items()}

In [ ]:
sum(mitigated_counts.values())

In [ ]:
ev_rem = get_lst_ev(mitigated_counts, tqdm(observable_elements), tqdm(stabilizer_elements))
print(ev_rem)

In [ ]:
counts_dd = all_counts_dd[0]
mitigated_quasi_probs = mitigator.quasi_probabilities(counts_dd)
mitigated_stddev = mitigated_quasi_probs._stddev_upper_bound
mitigated_probs = (mitigated_quasi_probs.nearest_probability_distribution().binary_probabilities())
mitigated_counts_dd = {k: round(v * nshots) for k, v in mitigated_probs.items()}

In [ ]:
sum(mitigated_counts_dd.values())

In [ ]:
ev_rem_dd = get_lst_ev(mitigated_counts_dd, tqdm(observable_elements), tqdm(stabilizer_elements))
print(ev_rem_dd)

## Save results

In [ ]:
save_key = f"bell_steane_code_zz_{computer.name}_{time_key}_job_raw_id_{job_physical.job_id()}_job_encoded_id_{job.job_id()}_job_encoded_dd_id_{job_dd.job_id()}_rem_calibration_id_{result.job_ids[0]}"

In [ ]:
import os

In [ ]:
os.mkdir(save_key)

In [ ]:
np.savetxt(f"{save_key}/qubits_physical.txt", layout[1])
np.savetxt(f"{save_key}/qubits_encoded.txt", layout[n])

In [ ]:
np.savetxt(f"{save_key}/nshots.txt", [nshots])

In [ ]:
np.savetxt(f"{save_key}/physical.txt", [ev_physical])
np.savetxt(f"{save_key}/encoded.txt", [ev])
np.savetxt(f"{save_key}/encoded_dd.txt", [ev_dd])
np.savetxt(f"{save_key}/encoded_rem.txt", [ev_rem])
np.savetxt(f"{save_key}/encoded_dd_rem.txt", [ev_rem_dd])